# Mitsui Commodity Challenge
Using GRU, Ridge, MLP, and Ensemble to predict commodity price


In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr
from scipy.optimize import minimize
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings("ignore")

TRAIN_CSV        = "train.csv"
TRAIN_LABELS_CSV = "train_labels.csv"
TEST_CSV         = "test.csv"
TEST_LABEL_FILES = {k: f"test_labels_lag_{k}.csv" for k in range(1, 5)}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

# Constants 
VAL_FRAC         = 0.15   # held-out 15% of training dates for validation
N_SPLITS_CV      = 5
TOP_N_PER_TARGET = 50
ALPHA_GRID       = [100, 500, 1000, 2000, 5000, 10000, 20000, 50000, 100000]
VOL_WINDOW       = 20
N_LAGS           = 4      # lag-1 through lag-4
MLP_EMB_DIM      = 128
GRU_EMB_DIM      = 256


Device: cpu


In [2]:
# =============================================================================
# SCORING — official competition metric: daily Spearman IC Sharpe ratio
# =============================================================================
def rank_correlation_sharpe_ratio(merged_df: pd.DataFrame) -> float:
    prediction_cols = [c for c in merged_df.columns if c.startswith("prediction_")]
    target_cols_eval = [c for c in merged_df.columns if c.startswith("target_")]

    def _row_rc(row):
        non_null = [c for c in target_cols_eval if not pd.isnull(row[c])]
        preds = [c for c in prediction_cols
                 if c.replace("prediction", "target") in non_null]
        if len(non_null) < 2: 
            return np.nan
        if row[non_null].std(ddof=0) == 0 or row[preds].std(ddof=0) == 0:
            return np.nan
        return np.corrcoef(
            row[preds].rank(method="average"),
            row[non_null].rank(method="average")
        )[0, 1]

    daily = merged_df.apply(_row_rc, axis=1).dropna()
    std = daily.std(ddof=0)
    return float(daily.mean() / std) if std > 0 else 0.0


def official_score(pred_arr: np.ndarray, Y_true: pd.DataFrame,
                   target_cols: list) -> float:
    pred_df = pd.DataFrame(pred_arr, columns=target_cols)
    sub = pred_df.rename(
        columns={c: c.replace("target_", "prediction_") for c in target_cols}
    )
    merged = pd.concat(
        [Y_true[target_cols].reset_index(drop=True),
         sub.reset_index(drop=True)],
        axis=1
    )
    return rank_correlation_sharpe_ratio(merged)


def eval_pearson(preds_arr, Y_true_df, obs_df, target_cols):
    preds_df = pd.DataFrame(preds_arr, columns=target_cols)
    scores = []
    for col in target_cols:
        obs_col = f"{col}_observed"
        if obs_col not in obs_df.columns: 
            continue
        mask = obs_df[obs_col].astype(bool).values
        if mask.sum() < 2: 
            continue
        r, _ = pearsonr(Y_true_df[col].values[mask], preds_df[col].values[mask])
        if np.isfinite(r): 
            scores.append(r)
    return float(np.mean(scores)) if scores else 0.0

In [3]:
# load training data (train.csv & train_labels.csv) & split using time-based 

train_feat = pd.read_csv(TRAIN_CSV).sort_values("date_id").reset_index(drop=True)
print(f"train data shape: {train_feat.shape}")

train_labs = pd.read_csv(TRAIN_LABELS_CSV).sort_values("date_id").reset_index(drop=True)
print(f"train label shape: {train_labs.shape}")

train_merged = train_feat.merge(
    train_labs[["date_id"] + [c for c in train_labs.columns if c.startswith("target_")]],
    on = "date_id", how = "inner"
).sort_values("date_id").reset_index(drop=True)

print(f"merged data shape: {train_merged.shape}")

target_cols = sorted([c for c in train_labs.columns if c.startswith("target_")])
n_tgts      = len(target_cols)
print(f"Targets: {n_tgts}")

# Time-based split — no shuffling
unique_dates  = np.sort(train_merged["date_id"].unique())
n_val_dates   = max(1, int(len(unique_dates) * VAL_FRAC))
val_dates_set = set(unique_dates[-n_val_dates:])
tr_dates_set  = set(unique_dates[:-n_val_dates])

train_tr  = train_merged[train_merged["date_id"].isin(tr_dates_set)].reset_index(drop = True)
train_val = train_merged[train_merged["date_id"].isin(val_dates_set)].reset_index(drop = True)

print(f"Train rows: {len(train_tr)}, date_range {train_tr['date_id'].min()} – {train_tr['date_id'].max()}")
print(f"Val rows: {len(train_val)}, date_range {train_val['date_id'].min()} – {train_val['date_id'].max()}")
if tr_dates_set & val_dates_set == set():
    print("Dates overlap between train and val")

meta_cols   = ["date_id"]
label_leak  = [c for c in train_merged.columns if c.startswith("label_target_")]
drop_always = set(meta_cols + label_leak + target_cols)
feat_cols   = [c for c in train_feat.columns if c not in drop_always]

X_train_raw  = train_tr[feat_cols].reset_index(drop = True)
X_val_raw    = train_val[feat_cols].reset_index(drop = True)
Y_train_lag1 = train_tr[target_cols].fillna(0.0).reset_index(drop = True)
Y_val_lag1   = train_val[target_cols].fillna(0.0).reset_index(drop = True)
n_train      = len(X_train_raw)
n_val        = len(X_val_raw)

obs_train = (train_tr[target_cols].notna()
             .rename(columns={c: f"{c}_observed" for c in target_cols})
             .reset_index(drop=True))
obs_val   = (train_val[target_cols].notna()
             .rename(columns={c: f"{c}_observed" for c in target_cols})
             .reset_index(drop=True))

print(f"# of feature cols: {len(feat_cols)}")

train data shape: (1961, 558)
train label shape: (1961, 425)
merged data shape: (1961, 982)
Targets: 424
Train rows: 1667, date_range 0 – 1666
Val rows: 294, date_range 1667 – 1960
Dates overlap between train and val
# of feature cols: 557


In [4]:
# Create auxiliary training targets by forward-shifting labels.
# Lag 1 is the real label; higher lags are synthetic pseudo-labels
Y_train_per_lag = {}
for k in range(1, N_LAGS + 1):
    shifted = train_tr[target_cols].shift(-k).fillna(0.0).reset_index(drop=True)
    Y_train_per_lag[k] = shifted
Y_train_per_lag[1] = Y_train_lag1.copy()  

for k in range(1, N_LAGS + 1):
    nz = (Y_train_per_lag[k].values != 0).mean()
    tag = "" if k == 1 else "  [forward-shifted pseudo-lag]"
    print(f"  Y_train lag{k}: {nz*100:.1f}% non-zero, rows: {len(Y_train_per_lag[k])}{tag}")

# Use the same true validation label for every lag head to avoid future-label leakage
Y_val_per_lag = {}
for k in range(1, N_LAGS + 1):
    Y_val_per_lag[k] = Y_val_lag1.copy()  

print(f"All {N_LAGS} lag targets constructed.")
print("Train: pseudo-lags via forward shift (regularization only).")
print("Val:   all lag slots = Y_val_lag1 (no future-label lookahead)")

  Y_train lag1: 89.5% non-zero, rows: 1667
  Y_train lag2: 89.4% non-zero, rows: 1667  [forward-shifted pseudo-lag]
  Y_train lag3: 89.4% non-zero, rows: 1667  [forward-shifted pseudo-lag]
  Y_train lag4: 89.3% non-zero, rows: 1667  [forward-shifted pseudo-lag]
All 4 lag targets constructed.
Train: pseudo-lags via forward shift (regularization only).
Val:   all lag slots = Y_val_lag1 (no future-label lookahead)


In [5]:
#fill missing value with 0
X_train_f = X_train_raw.fillna(0.0)
X_val_f   = X_val_raw.fillna(0.0)

miss_ratio        = X_train_raw.isna().mean()
high_missing_cols = miss_ratio[miss_ratio > 0.95].index.tolist()
low_var_cols      = X_train_f.var(axis=0)
low_var_cols      = low_var_cols[low_var_cols < 1e-10].index.tolist()
drop_cols         = sorted(set(high_missing_cols) | set(low_var_cols))
keep_cols         = [c for c in X_train_raw.columns if c not in drop_cols]

X_train_c = X_train_f[keep_cols].copy()
X_val_c   = X_val_f[keep_cols].copy()
print(f"Features: {len(keep_cols)} kept, {len(drop_cols)} dropped")
print(f"High-missing (>95%): {len(high_missing_cols)}, Low-var: {len(low_var_cols)}")


Features: 557 kept, 0 dropped
High-missing (>95%): 0, Low-var: 0


In [6]:
# Compute rolling target volatility from training labels and use it to normalize targets
def compute_rolling_vol(Y_df, window):
    vol = Y_df.rolling(window = window, min_periods = 5).std()
    return vol.bfill().fillna(1.0).replace(0, 1.0)

# Rescales each target into volatility units so periods with higher/lower variance are comparable
train_vol = compute_rolling_vol(Y_train_lag1, VOL_WINDOW)
val_vol_scalar = train_vol.iloc[-1].values  
val_vol_df = pd.DataFrame(
    np.tile(val_vol_scalar, (n_val, 1)), columns = target_cols
)

# Clip values to avoid extreme targets from dominating training
def normalize_targets(Y_df, vol_df):
    return np.clip((Y_df.values / vol_df.values).astype(np.float32), -5.0, 5.0)

Y_train_norm_per_lag = {}
for k in range(1, N_LAGS + 1):
    Y_train_norm_per_lag[k] = normalize_targets(Y_train_per_lag[k], train_vol)
    print(f"Lag {k} norm range: [{Y_train_norm_per_lag[k].min():.2f}, {Y_train_norm_per_lag[k].max():.2f}]")

# Validation uses the final training volatility level to avoid lookahead
Y_val_norm_per_lag = {}
for k in range(1, N_LAGS + 1):
    Y_val_norm_per_lag[k] = normalize_targets(Y_val_per_lag[k], val_vol_df)

print(f"All {N_LAGS} lags vol-normalised.")


Lag 1 norm range: [-5.00, 5.00]
Lag 2 norm range: [-5.00, 5.00]
Lag 3 norm range: [-5.00, 5.00]
Lag 4 norm range: [-5.00, 5.00]
All 4 lags vol-normalised.


In [7]:
# Convert targets to cross-sectional ranks within each timestamp so the model learns relative ordering
# rather than raw target magnitudes. Missing/unobserved targets are excluded from the ranking
def rank_transform_Y(Y_df: pd.DataFrame, obs_df=None) -> pd.DataFrame:
    Y_arr = Y_df.values.copy().astype(float)
    Y_rank = np.zeros_like(Y_arr)
    tgt_names = list(Y_df.columns)
    for t in range(Y_arr.shape[0]):
        if obs_df is not None:
            obs_here = [f"{c}_observed" for c in tgt_names if f"{c}_observed" in obs_df.columns]
            obs_row  = (obs_df[obs_here].iloc[t].values.astype(bool)
                        if obs_here else np.ones(len(tgt_names), dtype=bool))
        else:
            obs_row = (Y_arr[t] != 0)
        obs_idx = np.where(obs_row)[0]
        if len(obs_idx) < 2: 
            continue
        vals  = Y_arr[t, obs_idx]
        ranks = pd.Series(vals).rank(method="average").values
        n     = len(ranks)
        Y_rank[t, obs_idx] = 2 * (ranks - 1) / (n - 1) - 1
    return pd.DataFrame(Y_rank, columns=Y_df.columns)

Y_train_rank_per_lag = {}
for k in range(1, N_LAGS + 1):
    Y_train_rank_per_lag[k] = rank_transform_Y(Y_train_per_lag[k], obs_train)
    nz = (Y_train_rank_per_lag[k].values != 0).mean()
    print(f"Y_train_rank lag{k}: {nz*100:.1f}% non-zero")

# Add cross-sectional percentile-rank features for groups of related inputs
# (e.g. returns, rolling means, rolling stds) so the model can compare each item
# against the rest of the panel at the same timestamp.
def add_cross_sectional_ranks(X_df: pd.DataFrame) -> pd.DataFrame:
    rank_suffixes = ["_ret1", "_ret5", "_rmean5", "_rmean20", "_rstd5"]
    new_cols = {}
    for suf in rank_suffixes:
        group = [c for c in X_df.columns if c.endswith(suf)]
        if len(group) < 2: 
            continue
        ranked = X_df[group].rank(axis=1, pct=True)
        for col in group:
            new_cols[f"{col}_xrank"] = ranked[col]
    if not new_cols: 
        return X_df
    return pd.concat([X_df, pd.DataFrame(new_cols, index=X_df.index)], axis=1)

X_train_aug = add_cross_sectional_ranks(X_train_c)
X_val_aug   = add_cross_sectional_ranks(X_val_c)
print(f"X_rank features added: {X_train_aug.shape[1] - X_train_c.shape[1]}")

# Standardize all features (Ridge)
scaler_main = StandardScaler()
X_train_s   = scaler_main.fit_transform(X_train_aug).astype(np.float32)
X_val_s     = scaler_main.transform(X_val_aug).astype(np.float32)


Y_train_rank lag1: 89.5% non-zero
Y_train_rank lag2: 89.3% non-zero
Y_train_rank lag3: 89.2% non-zero
Y_train_rank lag4: 89.0% non-zero
X_rank features added: 0


In [8]:
# time series cross validation (Ridge)
def cv_alpha(X_s, Y_s, alpha_grid = ALPHA_GRID, n_splits = N_SPLITS_CV):
    tscv = TimeSeriesSplit(n_splits = n_splits)
    best_a, best_sc = alpha_grid[0], -np.inf
    for a in alpha_grid:
        fold_sps = []
        for tr_idx, vl_idx in tscv.split(X_s):
            m = Ridge(alpha = a, random_state = SEED)
            m.fit(X_s[tr_idx], Y_s[tr_idx])
            pred = m.predict(X_s[vl_idx])
            n_out = Y_s.shape[1] if Y_s.ndim > 1 else 1
            p2, y2 = pred.reshape(-1, n_out), Y_s[vl_idx].reshape(-1, n_out)
            sp = [float(spearmanr(y2[:, j], p2[:, j]).correlation)
                  for j in range(n_out)
                  if np.std(y2[:, j]) > 1e-12 and np.std(p2[:, j]) > 1e-12] # Skip targets with near-zero variance
            if sp: fold_sps.append(float(np.mean(sp)))
        sc = float(np.mean(fold_sps)) if fold_sps else -np.inf
        if sc > best_sc: best_sc, best_a = sc, a
    return best_a

In [9]:
# shared Ridge on 4 lags
print("EXP A — Shared Ridge (rank-Y, 4-lag avg)")

ridge_A_models = {}   # store fitted models for test reuse
lag_preds_A = {}

# Each lag gets its own tuned alpha, but all models share the same feature set
for k in range(1, N_LAGS + 1):
    Y_rk = Y_train_rank_per_lag[k].values.astype(np.float32)
    best_a = cv_alpha(X_train_s, Y_rk) 
    ridge = Ridge(alpha=best_a, random_state=SEED)
    ridge.fit(X_train_s, Y_rk)
    ridge_A_models[k] = ridge
    p_rank = ridge.predict(X_val_s).astype(np.float32) # rank scale
    p_ret  = p_rank * val_vol_scalar # return scale
    lag_preds_A[k] = p_ret
    sc_k = official_score(p_ret, Y_val_lag1, target_cols)
    print(f"Lag {k} (alpha = {best_a}): val_sharpe = {sc_k:.4f}")

# Average the 4 lag predictions for a more stable ensemble forecast
pred_A     = np.mean([lag_preds_A[k] for k in range(1, N_LAGS + 1)], axis = 0)
# Score the blended prediction against the true lag-1 validation target,
# because lag-1 is the real forecasting objective and other lags are only auxiliary
score_A    = official_score(pred_A, Y_val_lag1, target_cols)
print(f"Exp A (4-lag avg, lag-1 score): val_sharpe = {score_A:.4f}")
pred_A_all = np.stack([lag_preds_A[k] for k in range(1, N_LAGS + 1)], axis = 1) # keep all per-lag predictions for analysis
print(f"pred_A on all 4 lags shape: {pred_A_all.shape}")


EXP A — Shared Ridge (rank-Y, 4-lag avg)
Lag 1 (alpha = 100): val_sharpe = -0.0062
Lag 2 (alpha = 100): val_sharpe = -0.0202
Lag 3 (alpha = 100): val_sharpe = -0.0426
Lag 4 (alpha = 100): val_sharpe = -0.0281
Exp A (4-lag avg, lag-1 score): val_sharpe = -0.0189
pred_A on all 4 lags shape: (294, 4, 424)


In [10]:
# per-target ridge using top-50 features of each target (based on Pearson corr). fit 1 ridge per lag --> average out
print("EXP B — Per-target Ridge (top-50 corr, rank-Y, 4-lag avg)")

Xv = X_train_aug.values.astype(float)
X_c_ = Xv - Xv.mean(axis = 0, keepdims = True)
Y_raw = Y_train_lag1.values.astype(float)

pred_B = np.zeros((n_val, n_tgts), dtype = np.float32)         
pred_B_all = np.zeros((n_val, N_LAGS, n_tgts), dtype = np.float32)  

ridge_B_feature_idx = {} 
ridge_B_alpha = {}    

for j, tgt in enumerate(target_cols):
    y_raw = Y_raw[:, j]
    obs_col = f"{tgt}_observed"
    obs_mask = (
        obs_train[obs_col].astype(bool).values
        if obs_col in obs_train.columns else (y_raw != 0)
    )
    obs_idx_j = np.where(obs_mask)[0]

    # select top-N (n = 50) features per target by Pearson corr
    if len(obs_idx_j) < 10:
        top_idx = np.arange(min(TOP_N_PER_TARGET, Xv.shape[1]))
    else:
        y_obs = y_raw[obs_idx_j]
        y_c = y_obs - y_obs.mean()
        corr = (X_c_[obs_idx_j].T @ y_c) / (
            (X_c_[obs_idx_j].std(0, ddof=0) + 1e-12)
            * (y_c.std(ddof=0) + 1e-12) * len(obs_idx_j)
        )
        top_idx = np.argsort(-np.abs(corr))[:TOP_N_PER_TARGET]

    ridge_B_feature_idx[j] = top_idx 

    sel = [X_train_aug.columns[i] for i in top_idx]
    sc_X = StandardScaler()
    X_tr_s = sc_X.fit_transform(X_train_aug[sel].values)
    X_vl_s = sc_X.transform(X_val_aug[sel].values)

    # Tune alpha using lag-1 CV on rank-transformed Y
    y_rk1 = Y_train_rank_per_lag[1].values[:, j]
    sc_y = StandardScaler()
    y_rk1_s = sc_y.fit_transform(y_rk1.reshape(-1, 1)).ravel()

    tscv = TimeSeriesSplit(n_splits = N_SPLITS_CV)
    best_a_j, best_sc_j = ALPHA_GRID[0], -np.inf
    for a in ALPHA_GRID:
        fps = []
        for tr_i, vl_i in tscv.split(X_tr_s):
            m = Ridge(alpha = a, random_state = SEED).fit(X_tr_s[tr_i], y_rk1_s[tr_i])
            p, yt = m.predict(X_tr_s[vl_i]), y_rk1_s[vl_i]
            if np.std(yt) > 1e-12 and np.std(p) > 1e-12:
                fps.append(float(spearmanr(yt, p).correlation))
        sc_ = float(np.mean(fps)) if fps else -np.inf
        if sc_ > best_sc_j:
            best_sc_j, best_a_j = sc_, a

    ridge_B_alpha[j] = best_a_j 

    # 1 Ridge per lag --> store predictions in pred_B
    lag_preds_j = []
    for k in range(1, N_LAGS + 1):
        y_rk_k = Y_train_rank_per_lag[k].values[:, j]
        sc_yk = StandardScaler()
        y_rk_ks = sc_yk.fit_transform(y_rk_k.reshape(-1, 1)).ravel()
        m_k = Ridge(alpha = best_a_j, random_state = SEED).fit(X_tr_s, y_rk_ks)
        p_k = sc_yk.inverse_transform(m_k.predict(X_vl_s).reshape(-1, 1)).ravel()
        lag_preds_j.append(p_k)
        pred_B_all[:, k - 1, j] = p_k

    pred_B[:, j] = np.mean(lag_preds_j, axis=0)

    if j % 20 == 0:
        print(f" target {j:3d} / {n_tgts} alpha = {best_a_j}") # report alpha every 20 targets

score_B = official_score(pred_B, Y_val_lag1, target_cols)
print(f"Exp B (4-lag avg per target): val_sharpe = {score_B:.4f}")

EXP B — Per-target Ridge (top-50 corr, rank-Y, 4-lag avg)
 target   0 / 424 alpha = 100
 target  20 / 424 alpha = 500
 target  40 / 424 alpha = 50000
 target  60 / 424 alpha = 2000
 target  80 / 424 alpha = 1000
 target 100 / 424 alpha = 100
 target 120 / 424 alpha = 100000
 target 140 / 424 alpha = 100
 target 160 / 424 alpha = 1000
 target 180 / 424 alpha = 500
 target 200 / 424 alpha = 5000
 target 220 / 424 alpha = 100000
 target 240 / 424 alpha = 100000
 target 260 / 424 alpha = 5000
 target 280 / 424 alpha = 1000
 target 300 / 424 alpha = 5000
 target 320 / 424 alpha = 2000
 target 340 / 424 alpha = 20000
 target 360 / 424 alpha = 50000
 target 380 / 424 alpha = 500
 target 400 / 424 alpha = 50000
 target 420 / 424 alpha = 500
Exp B (4-lag avg per target): val_sharpe = 0.2499


In [11]:
# MLP encoder (multi-task, all 4 lags)
# Early-stop on full 4-lag val loss
# Report pearson as a secondary diagnostic metric to check whether predictions
# line up linearly with the targets, not for final evaluation

print("MLP ENCODER (4-lag multi-task)")

class MLPEncoder(nn.Module):
    def __init__(self, in_dim, out_dim, hidden = (512, 256)):
        super().__init__()
        layers = [nn.BatchNorm1d(in_dim)]
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers += [nn.Linear(prev, MLP_EMB_DIM), nn.ReLU()]
        self.encoder = nn.Sequential(*layers)
        self.head = nn.Linear(MLP_EMB_DIM, out_dim)

    def forward(self, x): 
        return self.head(self.encoder(x))
    
    def encode(self, x):  
        return self.encoder(x)


def train_nn(model, loader, X_vl_t, Y_vl_t,
             lr = 1e-3, wd = 1e-4, epochs = 80, patience = 10):
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    crit = nn.MSELoss()
    X_vl_t, Y_vl_t = X_vl_t.to(DEVICE), Y_vl_t.to(DEVICE)
    best_val, best_state, pat = float("inf"), None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        tr_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb), yb)
            loss.backward(); opt.step()
            tr_loss += loss.item() * len(xb)
        tr_loss /= len(loader.dataset)
        model.eval()
        with torch.no_grad():
            vl = crit(model(X_vl_t), Y_vl_t).item()
        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d}, train = {tr_loss:.5f}, val = {vl:.5f}")
        if vl < best_val:
            best_val = vl
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                print(f"Early stop @ epoch {epoch}, best_val = {best_val:.5f}")
                break
    model.load_state_dict(best_state)


# stack all 4 lags
Y_train_mlp = np.hstack([Y_train_norm_per_lag[k] for k in range(1, N_LAGS + 1)]).astype(np.float32)
Y_val_mlp   = np.hstack([Y_val_norm_per_lag[k]   for k in range(1, N_LAGS + 1)]).astype(np.float32)

X_tr_mlp_t = torch.tensor(X_train_s, dtype = torch.float32)
Y_tr_mlp_t = torch.tensor(Y_train_mlp, dtype = torch.float32)
X_vl_mlp_t = torch.tensor(X_val_s, dtype = torch.float32)
Y_vl_mlp_t = torch.tensor(Y_val_mlp, dtype = torch.float32)

mlp = MLPEncoder(X_train_s.shape[1], Y_train_mlp.shape[1]).to(DEVICE)
mlp_loader = DataLoader(TensorDataset(X_tr_mlp_t, Y_tr_mlp_t), batch_size = 64, shuffle = True)

print(f"MLP output dim: {Y_train_mlp.shape[1]} ({N_LAGS} lags × {n_tgts} targets)")
print(f"# MLP params: {sum(p.numel() for p in mlp.parameters()):,}")
train_nn(mlp, mlp_loader, X_vl_mlp_t, Y_vl_mlp_t)

mlp.eval()
with torch.no_grad():
    mlp_emb_tr  = mlp.encode(X_tr_mlp_t.to(DEVICE)).cpu().numpy()
    mlp_emb_vl  = mlp.encode(X_vl_mlp_t.to(DEVICE)).cpu().numpy()
    mlp_raw_vl  = mlp(X_vl_mlp_t.to(DEVICE)).cpu().numpy()  # (n_val, N_LAGS * n_tgts)

scale_4lags_val   = np.tile(val_vol_scalar, N_LAGS)
mlp_preds_all_val = (mlp_raw_vl * scale_4lags_val).reshape(-1, N_LAGS, n_tgts)
mlp_preds_lag1    = mlp_preds_all_val[:, 0, :]  

score_mlp = official_score(mlp_preds_lag1, Y_val_lag1, target_cols)
pear_mlp  = eval_pearson(mlp_preds_lag1, Y_val_lag1, obs_val, target_cols)
print(f"MLP: val Sharpe (lag-1): {score_mlp:.4f}, Pearson: {pear_mlp:.4f}")

MLP ENCODER (4-lag multi-task)
MLP output dim: 1696 (4 lags × 424 targets)
# MLP params: 669,818
Epoch  10, train = 1.30599, val = 1.07259
Early stop @ epoch 17, best_val = 1.06400
MLP: val Sharpe (lag-1): 0.2472, Pearson: 0.0277


In [12]:
# multi-scale GRU encoders (windows 4 and 10, all 4 lags)

print("MULTI-SCALE GRU ENCODERS (ALL 4 LAGS)")

# Exclude engineered suffixes so GRU learns its own lags
ENG_SUFFIXES = (
    "_lag1","_lag2","_lag3","_lag5",
    "_rmean5","_rmean10","_rmean20",
    "_rstd5","_rstd10","_rstd20",
    "_rmax5","_rmax10","_rmax20",
    "_rmin5","_rmin10","_rmin20",
    "_ret1","_ret5","_mom20",
    "_spread","_ratio","_zscore","_xrank",
)
raw_cols = [
    c for c in X_train_c.columns
    if not any(c.endswith(s) for s in ENG_SUFFIXES)
    and not c.startswith("label_")
    and not c.endswith("_observed")
]
print(f"Raw cols for GRU: {len(raw_cols)}")

raw_scaler = StandardScaler()
X_raw_tr   = raw_scaler.fit_transform(X_train_c[raw_cols]).astype(np.float32)
X_raw_vl   = raw_scaler.transform(X_val_c[raw_cols]).astype(np.float32)
N_RAW      = X_raw_tr.shape[1]

Y_gru_tr = np.hstack([Y_train_norm_per_lag[k] for k in range(1, N_LAGS + 1)]).astype(np.float32)
Y_gru_vl = np.hstack([Y_val_norm_per_lag[k] for k in range(1, N_LAGS + 1)]).astype(np.float32)


class GRUEncoder(nn.Module):
    def __init__(self, in_dim, out_dim, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(in_dim, GRU_EMB_DIM, num_layers = num_layers,
                          batch_first = True,
                          dropout = dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Linear(GRU_EMB_DIM, 128), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, out_dim)
        )

    def forward(self, x):
        i, h = self.gru(x)
        return self.head(h[-1])

    def encode(self, x):
        i, h = self.gru(x)
        return h[-1]

def make_windows(X, Y, window):
    Xs, Ys = [], []
    for i in range(window - 1, len(X)):
        Xs.append(X[i - window + 1: i + 1])
        Ys.append(Y[i])
    return np.array(Xs), np.array(Ys)

def train_gru_scale(window):
    print(f"\n--- GRU window = {window} ---")
    offset = window - 1
    X_seq_tr, Y_seq_tr = make_windows(X_raw_tr, Y_gru_tr, window)
    # Warmup val sequence with window-1 training feature rows
    X_border  = np.vstack([X_raw_tr[-offset:], X_raw_vl])
    Y_val_pad = np.vstack([np.zeros((offset, Y_gru_vl.shape[1]), dtype = np.float32), Y_gru_vl])
    X_seq_vl, Y_seq_vl = make_windows(X_border, Y_val_pad, window)

    print(f"Train seq: {X_seq_tr.shape}, Val seq: {X_seq_vl.shape}")

    X_tr_t = torch.tensor(X_seq_tr, dtype=torch.float32)
    Y_tr_t = torch.tensor(Y_seq_tr, dtype=torch.float32)
    X_vl_t = torch.tensor(X_seq_vl, dtype=torch.float32)
    Y_vl_t = torch.tensor(Y_seq_vl, dtype=torch.float32)

    loader = DataLoader(TensorDataset(X_tr_t, Y_tr_t), batch_size = 32, shuffle = True)
    model  = GRUEncoder(N_RAW, Y_gru_tr.shape[1]).to(DEVICE)
    print(f"# Params: {sum(p.numel() for p in model.parameters()):,}")
    train_nn(model, loader, X_vl_t, Y_vl_t, lr = 5e-4, wd = 1e-4)

    model.eval()
    with torch.no_grad():
        preds_norm_all = model(X_vl_t.to(DEVICE)).cpu().numpy()  # (n_val, N_LAGS*n_tgts)
        emb_tr         = model.encode(X_tr_t.to(DEVICE)).cpu().numpy()
        emb_vl         = model.encode(X_vl_t.to(DEVICE)).cpu().numpy()

    preds_all  = (preds_norm_all * scale_4lags_val).reshape(-1, N_LAGS, n_tgts)
    preds_lag1 = preds_all[:, 0, :]

    sc = official_score(preds_lag1, Y_val_lag1, target_cols)
    pr = eval_pearson(preds_lag1, Y_val_lag1, obs_val, target_cols)
    print(f"GRU w = {window}: val Sharpe (lag-1): {sc:.4f}, Pearson: {pr:.4f}")
    return model, preds_lag1, preds_all, emb_tr, emb_vl, offset, sc

gru4,  gru4_lag1,  gru4_all,  gru4_emb_tr,  gru4_emb_vl,  offset4,  score_gru4  = train_gru_scale(4)
gru10, gru10_lag1, gru10_all, gru10_emb_tr, gru10_emb_vl, offset10, score_gru10 = train_gru_scale(10)

gru_avg_all  = (gru4_all + gru10_all) / 2.0
gru_avg_lag1 = gru_avg_all[:, 0, :]
score_gru_avg = official_score(gru_avg_lag1, Y_val_lag1, target_cols)
print(f"\nGRU avg (w4+w10): val Sharpe (lag-1): {score_gru_avg:.4f}")


MULTI-SCALE GRU ENCODERS (ALL 4 LAGS)
Raw cols for GRU: 557

--- GRU window = 4 ---
Train seq: (1664, 4, 557), Val seq: (294, 4, 557)
# Params: 1,272,352
Epoch  10, train = 1.12557, val = 1.23697
Early stop @ epoch 12, best_val = 1.06482
GRU w = 4: val Sharpe (lag-1): 0.2803, Pearson: 0.0379

--- GRU window = 10 ---
Train seq: (1658, 10, 557), Val seq: (294, 10, 557)
# Params: 1,272,352
Epoch  10, train = 1.06742, val = 1.13387
Early stop @ epoch 12, best_val = 1.06517
GRU w = 10: val Sharpe (lag-1): 0.3332, Pearson: 0.0337

GRU avg (w4+w10): val Sharpe (lag-1): 0.3461


In [13]:
# Use the embeddings from the GRU and MLP models as inputs to a final Ridge model.
# First align all embeddings to the same rows, then tune Ridge using lag-1 validation score.
# Train one Ridge model for each lag, and test whether adding more embeddings
# or raw features improves the final prediction.

print("RIDGE META-MODELS ON EMBEDDINGS — MULTI-LAG")

OFFSET = offset10   # align all embeddings to the shortest (GRU-10) sequence

gru4_emb_tr_a = gru4_emb_tr[OFFSET - offset4:]
gru10_emb_tr_a = gru10_emb_tr
mlp_emb_tr_a = mlp_emb_tr[OFFSET:]
X_train_s_a = X_train_s[OFFSET:]

n_aligned = min(gru4_emb_tr_a.shape[0], gru10_emb_tr_a.shape[0],
                mlp_emb_tr_a.shape[0], X_train_s_a.shape[0])
gru4_emb_tr_a = gru4_emb_tr_a[:n_aligned]
gru10_emb_tr_a = gru10_emb_tr_a[:n_aligned]
mlp_emb_tr_a = mlp_emb_tr_a[:n_aligned]
X_train_s_a = X_train_s_a[:n_aligned]
print(f"# of aligned train rows: {n_aligned}")

ridge_store_all = {}  

def fit_ridge_multi_lag(name, X_tr, X_vl):
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_vl_s = scaler.transform(X_vl)

    Y_tr_lag1_aligned = Y_train_norm_per_lag[1][OFFSET:OFFSET + n_aligned]
    best_a, best_sc   = ALPHA_GRID[0], -np.inf
    for alpha in ALPHA_GRID:
        r = Ridge(alpha = alpha, random_state=SEED).fit(X_tr_s, Y_tr_lag1_aligned)
        p = r.predict(X_vl_s).astype(np.float32) * val_vol_scalar
        sc_v = official_score(p, Y_val_lag1, target_cols)
        if sc_v > best_sc:
            best_sc, best_a = sc_v, alpha

    preds_all  = np.zeros((X_vl.shape[0], N_LAGS, n_tgts), dtype=np.float32)
    model_list = []
    for k in range(1, N_LAGS + 1):
        Y_tr_k = Y_train_norm_per_lag[k][OFFSET:OFFSET + n_aligned]
        mdl = Ridge(alpha = best_a, random_state = SEED).fit(X_tr_s, Y_tr_k)
        pred_k = mdl.predict(X_vl_s).astype(np.float32) * val_vol_scalar
        preds_all[:, k - 1, :] = pred_k
        model_list.append(mdl)

    ridge_store_all[name] = (scaler, model_list)
    sc_lag1 = official_score(preds_all[:, 0, :], Y_val_lag1, target_cols)
    print(f"{name:<28} alpha = {best_a:>7}: val_sharpe(lag-1) = {sc_lag1:.4f}")
    return preds_all

X_gru_only_tr = np.hstack([gru4_emb_tr_a, gru10_emb_tr_a]).astype(np.float32)
X_gru_only_vl = np.hstack([gru4_emb_vl, gru10_emb_vl]).astype(np.float32)

X_gru_mlp_tr = np.hstack([gru4_emb_tr_a, gru10_emb_tr_a, mlp_emb_tr_a]).astype(np.float32)
X_gru_mlp_vl = np.hstack([gru4_emb_vl, gru10_emb_vl, mlp_emb_vl]).astype(np.float32)

X_mega_tr = np.hstack([gru4_emb_tr_a, gru10_emb_tr_a, mlp_emb_tr_a, X_train_s_a]).astype(np.float32)
X_mega_vl = np.hstack([gru4_emb_vl, gru10_emb_vl, mlp_emb_vl, X_val_s]).astype(np.float32)

ridge_gru_all = fit_ridge_multi_lag("R1: GRU emb only", X_gru_only_tr, X_gru_only_vl)
ridge_gru_mlp_all = fit_ridge_multi_lag("R2: GRU + MLP emb", X_gru_mlp_tr, X_gru_mlp_vl)
ridge_mega_all = fit_ridge_multi_lag("R3: GRU + MLP + raw", X_mega_tr, X_mega_vl)


RIDGE META-MODELS ON EMBEDDINGS — MULTI-LAG
# of aligned train rows: 1658
R1: GRU emb only             alpha =  100000: val_sharpe(lag-1) = 0.2480
R2: GRU + MLP emb            alpha =  100000: val_sharpe(lag-1) = 0.2574
R3: GRU + MLP + raw          alpha =  100000: val_sharpe(lag-1) = 0.2161


In [14]:
print("ENSEMBLE — ROBUST MULTI-LAG WEIGHT SELECTION")

def official_score_multilags(pred_3d, Y_true_per_lag, target_cols):
    """Average official_score across all 4 lag horizons."""
    return float(np.mean([
        official_score(pred_3d[:, k - 1, :], Y_true_per_lag[k], target_cols)
        for k in range(1, N_LAGS + 1)
    ]))

candidates_all = {
    "GRU_w4":           gru4_all,
    "GRU_w10":          gru10_all,
    "GRU_avg":          gru_avg_all,
    "MLP":              mlp_preds_all_val,
    "Ridge_A":          pred_A_all,
    "Ridge_per_target": pred_B_all,
    "Ridge_GRU":        ridge_gru_all,
    "Ridge_GRU_MLP":    ridge_gru_mlp_all,
    "Ridge_mega":       ridge_mega_all,
}

print(f"{'Model':<22} {'Val Sharpe (lag-1)':>20} {'Val Sharpe (all-lag avg)':>26}")
print("-" * 70)
full_scores = {}

for name, arr in candidates_all.items():
    sc_l1 = official_score(arr[:, 0, :], Y_val_lag1, target_cols)
    sc_all = official_score_multilags(arr, Y_val_per_lag, target_cols)
    full_scores[name] = sc_all
    flag = "** negative, excluded" if sc_all < 0 else ""
    print(f"{name:<22} {sc_l1:>20.4f} {sc_all:>26.4f}{flag}")

# Step 1: Only choose models with positive sharpe val
passing = {n: arr for n, arr in candidates_all.items() if full_scores[n] >= 0}
print(f"\nPassing model candidates: {list(passing.keys())}")

cand_names  = list(passing.keys())
cand_arrays = [passing[n] for n in cand_names]
n_cands     = len(cand_names)

# Step 2: Split val into tune (60%) and held-out (40%)
TUNE_FRAC = 0.6
tune_end  = int(n_val * TUNE_FRAC)

Y_tune_per_lag = {k: Y_val_per_lag[k].iloc[:tune_end].reset_index(drop=True)
                  for k in range(1, N_LAGS + 1)}
Y_held_per_lag = {k: Y_val_per_lag[k].iloc[tune_end:].reset_index(drop=True)
                  for k in range(1, N_LAGS + 1)}

arr_tune = [a[:tune_end] for a in cand_arrays]
arr_held = [a[tune_end:] for a in cand_arrays]

def score_tune_multilags(weights):
    weights = np.array(weights, dtype=float)
    weights = np.clip(weights, 0, None)
    s = weights.sum()
    if s < 1e-12: 
        return -np.inf
    weights /= s
    ens = sum(w * a for w, a in zip(weights, arr_tune))
    return official_score_multilags(ens, Y_tune_per_lag, target_cols)

def score_held_multilags(weights):
    weights = np.array(weights, dtype=float)
    weights = np.clip(weights, 0, None)
    s = weights.sum()
    if s < 1e-12: return -np.inf
    weights /= s
    ens = sum(w * a for w, a in zip(weights, arr_held))
    return official_score_multilags(ens, Y_held_per_lag, target_cols)

# Strategy A: Equal weights
w_A = np.ones(n_cands) / n_cands
sc_A = score_tune_multilags(w_A)
print(f"\nStrategy A (equal, {n_cands} candidates): tune = {sc_A:.4f}")

# Strategy B: Pick the 3 strongest models by individual validation score, then average those 3 equally as a simple ensemble.
tune_ind = {n: score_tune_multilags([1.0 if m == n else 0.0 for m in cand_names])
            for n in cand_names}
top3 = sorted(tune_ind, key = tune_ind.get, reverse = True)[:3]
w_B  = np.array([1/3 if n in top3 else 0.0 for n in cand_names])
sc_B = score_tune_multilags(w_B)
print(f"Strategy B (top-3 equal): tune = {sc_B:.4f} models = {top3}")

# Strategy C: Add Soft-max weight to each cand models
tune_scores_arr = np.array([tune_ind[n] for n in cand_names])
exp_s = np.exp(tune_scores_arr - tune_scores_arr.max())
w_C = exp_s / exp_s.sum()
sc_C = score_tune_multilags(w_C)
print(f"Strategy C (softmax-weighted): tune = {sc_C:.4f}")

# Step 4: Compare 3 strategies results
strategies = {"A_equal": (sc_A, w_A), "B_top3": (sc_B, w_B), "C_softmax": (sc_C, w_C)}
best_name  = max(strategies, key = lambda k: strategies[k][0])
best_sc_tune, w_final_passing = strategies[best_name]
w_final_passing = np.array(w_final_passing)
w_final_passing /= w_final_passing.sum()

print(f"\nBEST STRATEGY: Strategy {best_name}, tune val = {best_sc_tune:.4f}")
print(f"Weights:")
for n, w in zip(cand_names, w_final_passing):
    if w > 0.001: print(f"{n:<22}: {w:.3f}")

ensemble_all      = sum(w * a for w, a in zip(w_final_passing, cand_arrays))
best_sc_lag1      = official_score(ensemble_all[:, 0, :], Y_val_lag1, target_cols)
best_sc_tune_full = score_tune_multilags(w_final_passing)
best_sc_held_full = score_held_multilags(w_final_passing) 

print(f"\nEnsemble on full val:")
print(f"Val Sharpe (lag-1): {best_sc_lag1:.4f}")
print(f"Val Sharpe (tune 60%, all-lag): {best_sc_tune_full:.4f}")
print(f"Val Sharpe (held-out 40%): {best_sc_held_full:.4f}")

w_final = np.zeros(len(candidates_all))
all_names = list(candidates_all.keys())
for i, n in enumerate(all_names):
    if n in cand_names:
        j = cand_names.index(n)
        w_final[i] = w_final_passing[j]
cand_names_all = all_names

ENSEMBLE — ROBUST MULTI-LAG WEIGHT SELECTION
Model                    Val Sharpe (lag-1)   Val Sharpe (all-lag avg)
----------------------------------------------------------------------
GRU_w4                               0.2803                     0.2872
GRU_w10                              0.3332                     0.3009
GRU_avg                              0.3461                     0.3281
MLP                                  0.2472                     0.2583
Ridge_A                             -0.0062                    -0.0242** negative, excluded
Ridge_per_target                     0.2443                     0.2401
Ridge_GRU                            0.2480                     0.2460
Ridge_GRU_MLP                        0.2574                     0.2499
Ridge_mega                           0.2161                     0.2179

Passing model candidates: ['GRU_w4', 'GRU_w10', 'GRU_avg', 'MLP', 'Ridge_per_target', 'Ridge_GRU', 'Ridge_GRU_MLP', 'Ridge_mega']

Strategy A (equal, 8 

In [15]:
print("VALIDATION PERFORMANCE SUMMARY")

print(f"{'Model':<22} {'Val Sharpe (lag-1)':>20} {'Val Sharpe (all-lag)':>22} {'Weight':>8}")
print("-" * 76)

for i, (name, arr) in enumerate(candidates_all.items()):
    sl1  = official_score(arr[:, 0, :], Y_val_lag1, target_cols)
    sall = full_scores[name]
    w    = w_final[i]
    excl = "[excluded]" if w == 0 else ""
    print(f"{name:<22} {sl1:>20.4f} {sall:>22.4f} {w:>8.3f}{excl}")
print("-" * 76)
print(f"{'ENSEMBLE':<22} {best_sc_lag1:>20.4f} {best_sc_held_full:>22.4f}")

print(f"\nVal split: last {VAL_FRAC*100:.0f}% of training dates (no shuffle)")
print(f"Ensemble strategy: {best_name} — selected by held-out 40%% of val")


VALIDATION PERFORMANCE SUMMARY
Model                    Val Sharpe (lag-1)   Val Sharpe (all-lag)   Weight
----------------------------------------------------------------------------
GRU_w4                               0.2803                 0.2872    0.333
GRU_w10                              0.3332                 0.3009    0.333
GRU_avg                              0.3461                 0.3281    0.333
MLP                                  0.2472                 0.2583    0.000[excluded]
Ridge_A                             -0.0062                -0.0242    0.000[excluded]
Ridge_per_target                     0.2443                 0.2401    0.000[excluded]
Ridge_GRU                            0.2480                 0.2460    0.000[excluded]
Ridge_GRU_MLP                        0.2574                 0.2499    0.000[excluded]
Ridge_mega                           0.2161                 0.2179    0.000[excluded]
------------------------------------------------------------------------

In [16]:
# reconstruct test ground truth
n_targets = len(target_cols)
# load label files for all 4 lags, sorted by label_date_id and store in lag_files ls
lag_files = []
for k in range(1, N_LAGS + 1):
    df_k = (
        pd.read_csv(TEST_LABEL_FILES[k])
        .sort_values("label_date_id")
        .reset_index(drop=True)
    )
    lag_files.append(df_k)

# Use lag-1 as reference for test dates
ref_df = lag_files[0] #lag-1
test_label_date_ids = ref_df["label_date_id"].values
n_test = len(test_label_date_ids)
print(f"# of dates: {n_test}")
print(f"label_date_id range: {test_label_date_ids.min()} – {test_label_date_ids.max()}")

# Sanity check: no overlap of training dates in test
train_date_ids = set(train_merged["date_id"].values)
test_overlap = set(test_label_date_ids) & train_date_ids
if len(test_overlap) == 0:
    print("no overlap")

# Initialize final ground truth matrix: n_test dates x 424 targets
Y_test_true = pd.DataFrame(
    np.nan,
    index = np.arange(n_test),
    columns = target_cols,
)

# Fill in each lag block
for k, df_k in enumerate(lag_files, start = 1):
    tgt_cols_k = [c for c in df_k.columns if c.startswith("target_")]
    if not tgt_cols_k:
        continue
    # Map target names with indices within target_cols
    idxs = [target_cols.index(c) for c in tgt_cols_k]
    # sort cols to ensure right order 
    sorted_pairs = sorted(zip(tgt_cols_k, idxs), key = lambda x: int(x[0].split("_")[1]))
    tgt_cols_k_s = [p[0] for p in sorted_pairs]
    idxs_s = [p[1] for p in sorted_pairs]
    Y_test_true.iloc[:, idxs_s] = df_k[tgt_cols_k_s].values

print("ground truth file shape:", Y_test_true.shape)

gt_export = Y_test_true.copy()
ordered_cols = sorted(target_cols, key=lambda x: int(x.split("_")[1]))
gt_export = gt_export[ordered_cols]
gt_export.insert(0, "label_date_id", test_label_date_ids)
out_path = "ground_truths.csv"
gt_export.to_csv(out_path, index = False)

# of dates: 134
label_date_id range: 1827 – 1960
ground truth file shape: (134, 424)


In [17]:
test_raw = pd.read_csv(TEST_CSV).sort_values("date_id").reset_index(drop = True)

# Filter to rows whose date_id matches a label_date_id
test_raw_aligned = test_raw[test_raw["date_id"].isin(test_label_date_ids)].reset_index(drop=True)
if len(test_raw_aligned) < n_test:
    print(f"{n_test - len(test_raw_aligned)} label rows have no matching feature row. Filled with zeros")
test_feat_rows = test_raw_aligned

def preprocess_test(df):
    out = df.copy()
    for c in keep_cols:
        if c not in out.columns:
            out[c] = 0.0
    tc = out[keep_cols].fillna(0.0)          
    taug = add_cross_sectional_ranks(tc)    
    taug = taug.reindex(columns = X_train_aug.columns, fill_value = 0.0)
    ts = scaler_main.transform(taug).astype(np.float32)
    return ts, tc, taug

X_test_s, X_test_c, X_test_aug = preprocess_test(test_feat_rows)
print(f"Test feature matrix: {X_test_s.shape}")
print(f"Test aug matrix: {X_test_aug.shape}")

n_eval = min(len(X_test_s), n_test)
X_test_s = X_test_s[:n_eval]
X_test_c = X_test_c.iloc[:n_eval]
X_test_aug = X_test_aug.iloc[:n_eval]
Y_test_true = Y_test_true.iloc[:n_eval].reset_index(drop=True)
print(f"Evaluation rows: {n_eval}")


Test feature matrix: (134, 557)
Test aug matrix: (134, 557)
Evaluation rows: 134


In [18]:
print("TEST INFERENCE")

scale_4lags = np.tile(val_vol_scalar, N_LAGS)   

mlp.eval()
with torch.no_grad():
    mlp_raw_test = mlp(torch.tensor(X_test_s, dtype=torch.float32).to(DEVICE)).cpu().numpy()
mlp_test_all = (mlp_raw_test * scale_4lags).reshape(-1, N_LAGS, n_tgts)
print(f"MLP test all-lag shape: {mlp_test_all.shape}")

def make_test_windows(X_warmup, X_test, window):
    border = np.vstack([X_warmup[-(window - 1):], X_test])
    n = X_test.shape[0]
    return np.array([border[i - window + 1:i + 1] for i in range(window - 1, window - 1 + n)])

test_raw_mat = X_test_c.reindex(columns=raw_cols, fill_value=0.0).values.astype(np.float32)
X_raw_test = raw_scaler.transform(test_raw_mat).astype(np.float32)
X_seq_test4 = make_test_windows(X_raw_tr, X_raw_test, 4)
X_seq_test10 = make_test_windows(X_raw_tr, X_raw_test, 10)

gru4.eval()
gru10.eval()

with torch.no_grad():
    gru4_raw_test = gru4(torch.tensor(X_seq_test4,  dtype=torch.float32).to(DEVICE)).cpu().numpy()
    gru10_raw_test = gru10(torch.tensor(X_seq_test10, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    gru4_emb_test = gru4.encode(torch.tensor(X_seq_test4,  dtype=torch.float32).to(DEVICE)).cpu().numpy()
    gru10_emb_test = gru10.encode(torch.tensor(X_seq_test10, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    mlp_emb_test = mlp.encode(torch.tensor(X_test_s, dtype=torch.float32).to(DEVICE)).cpu().numpy()

gru4_test_all = (gru4_raw_test  * scale_4lags).reshape(-1, N_LAGS, n_tgts)
gru10_test_all = (gru10_raw_test * scale_4lags).reshape(-1, N_LAGS, n_tgts)
gru_avg_test_all = (gru4_test_all + gru10_test_all) / 2.0
print(f"avg gru_4 and gru_10 shape: {gru4_test_all.shape}")

def ridge_predict_test_all(name, X_test_mat):
    scaler, mdl_list = ridge_store_all[name]
    Xs = scaler.transform(X_test_mat)
    preds = np.zeros((X_test_mat.shape[0], N_LAGS, n_tgts), dtype=np.float32)
    for k, mdl in enumerate(mdl_list, start=1):
        preds[:, k - 1, :] = mdl.predict(Xs).astype(np.float32) * val_vol_scalar
    return preds

X_gru_only_test = np.hstack([gru4_emb_test, gru10_emb_test]).astype(np.float32)
X_gru_mlp_test = np.hstack([gru4_emb_test, gru10_emb_test, mlp_emb_test]).astype(np.float32)
X_mega_test = np.hstack([gru4_emb_test, gru10_emb_test, mlp_emb_test, X_test_s]).astype(np.float32)

ridge_gru_test_all = ridge_predict_test_all("R1: GRU emb only", X_gru_only_test)
ridge_gru_mlp_test_all = ridge_predict_test_all("R2: GRU + MLP emb", X_gru_mlp_test)
ridge_mega_test_all = ridge_predict_test_all("R3: GRU + MLP + raw", X_mega_test)

pred_A_test_all = np.zeros((n_eval, N_LAGS, n_tgts), dtype=np.float32)
for k in range(1, N_LAGS + 1):
    p_rank = ridge_A_models[k].predict(X_test_s).astype(np.float32)   
    pred_A_test_all[:, k - 1, :] = p_rank * val_vol_scalar            

pred_B_test_all = np.zeros((n_eval, N_LAGS, n_tgts), dtype=np.float32)

for j, tgt in enumerate(target_cols):
    top_idx = ridge_B_feature_idx[j]
    sel = [X_train_aug.columns[i] for i in top_idx]

    sc_X = StandardScaler()
    X_tr_s = sc_X.fit_transform(X_train_aug[sel].values)
    X_te_s = sc_X.transform(X_test_aug[sel].values)

    best_a_j = ridge_B_alpha[j]

    lag_preds_j = []
    for k in range(1, N_LAGS + 1):
        y_rk_k = Y_train_rank_per_lag[k].values[:, j]
        sc_yk = StandardScaler()
        y_rk_ks = sc_yk.fit_transform(y_rk_k.reshape(-1, 1)).ravel()
        m_k = Ridge(alpha=best_a_j, random_state=SEED).fit(X_tr_s, y_rk_ks)
        p_k = sc_yk.inverse_transform(m_k.predict(X_te_s).reshape(-1, 1)).ravel()
        pred_B_test_all[:, k - 1, j] = p_k

test_candidate_map_all = {
    "GRU_w4":           gru4_test_all,
    "GRU_w10":          gru10_test_all,
    "GRU_avg":          gru_avg_test_all,
    "MLP":              mlp_test_all,
    "Ridge_A":          pred_A_test_all,
    "Ridge_per_target": pred_B_test_all,
    "Ridge_GRU":        ridge_gru_test_all,
    "Ridge_GRU_MLP":    ridge_gru_mlp_test_all,
    "Ridge_mega":       ridge_mega_test_all,
}

missing = [n for n in cand_names_all if n not in test_candidate_map_all]
if missing:
    raise ValueError(f"Missing test predictions for: {missing}")

final_test_preds_all = sum(w * test_candidate_map_all[n] for w, n in zip(w_final, cand_names_all))
print(f"Final ensemble all-lag shape: {final_test_preds_all.shape}")


TEST INFERENCE
MLP test all-lag shape: (134, 4, 424)
avg gru_4 and gru_10 shape: (134, 4, 424)
Final ensemble all-lag shape: (134, 4, 424)


In [19]:

print("OFFLINE TEST EVALUATION USING GROUND TRUTHS")

n_eval = n_test  
print(f"Test rows with labels: {n_eval}")

pred_test_full_all = final_test_preds_all[:n_eval]                      # (n_eval, N_LAGS, 424)
y_true_eval        = Y_test_true.iloc[:n_eval].reset_index(drop=True)   # (n_eval, 424)

n_stored_lags = pred_test_full_all.shape[1]
lag_scores = []

for k, df_k in enumerate(lag_files, start=1):
    tgt_cols_k = [c for c in df_k.columns if c.startswith("target_")]
    lag_idx = k - 1
    p_k_full = pred_test_full_all[:, lag_idx, :]               
    idxs_k    = [target_cols.index(c) for c in tgt_cols_k]
    p_k_block = p_k_full[:, idxs_k]
    y_k_block = y_true_eval[tgt_cols_k].reset_index(drop=True)
    sc_k = official_score(p_k_block, y_k_block, tgt_cols_k)
    lag_scores.append(sc_k)
    print(
        f"Test Sharpe (lag-{k}, {len(tgt_cols_k)} targets): {sc_k:.4f}"
    )

test_sharpe_all_lag = float(np.mean(lag_scores)) if lag_scores else float("nan")
print(f"\nOffline test Sharpe (average over {len(lag_scores)} lag blocks): {test_sharpe_all_lag:.4f}")

print("="*60)
print(f"Validation Sharpe (lag-1, time-based split): {best_sc_lag1:.4f}")
print(f"Validation Sharpe (held-out 40% of val): {best_sc_held_full:.4f}")
print(f"Offline Test Sharpe (lag-block avg): {test_sharpe_all_lag:.4f}")


OFFLINE TEST EVALUATION USING GROUND TRUTHS
Test rows with labels: 134
Test Sharpe (lag-1, 106 targets): 0.1300
Test Sharpe (lag-2, 106 targets): 0.3004
Test Sharpe (lag-3, 106 targets): 0.2735
Test Sharpe (lag-4, 106 targets): 0.2991

Offline test Sharpe (average over 4 lag blocks): 0.2507
Validation Sharpe (lag-1, time-based split): 0.3461
Validation Sharpe (held-out 40% of val): 0.2720
Offline Test Sharpe (lag-block avg): 0.2507
